---
title: "Stage 4: Advanced MLflow Training"
author: "Abdelhamed Nour"
description: "Actual PyTorch training with MLflow tracking, artifacts, model logging, and optional registration"
format:
  html:
    toc: true
    code-fold: false
    code-tools: true
execute:
  eval: false
  echo: true
---

## Goal

This tutorial completes the MLflow path for the segmentation demo:

- run actual training through `train.py`
- log parameters and metrics
- save and log checkpoint artifacts
- log input / ground-truth / prediction image previews
- log a PyTorch model artifact
- optionally register the model
- inspect everything in the MLflow UI

The run is still tiny because this is a workflow tutorial, not an accuracy benchmark.

## Use The Same Runner

The key production-ML idea is that MLflow does not need a separate notebook-only training path.

The same agnostic runner should work for:

- local development
- DVC stages
- CI smoke tests
- MLflow tracked training
- later scheduled jobs

In [ ]:
from pathlib import Path

from ipcv.workflow import TrainingConfig, load_config, run_training

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "params.yaml").exists() else Path.cwd().parent
config = load_config(PROJECT_ROOT / "params.yaml").with_overrides(
    mlflow_tracking_uri=f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
)
config

## Fast MLflow Smoke Run

Use synthetic data first when validating logging behavior. This proves that MLflow tracking, artifacts, and model logging work without downloading VOC.

In [ ]:
smoke_config = config.with_overrides(
    run_name="mlflow-synthetic-smoke",
    use_synthetic_data=True,
    model_name="tiny",
    image_size=32,
    max_samples=4,
    batch_size=2,
    max_batches=1,
    checkpoint_path=str(PROJECT_ROOT / "models" / "mlflow-synthetic-smoke.pt"),
    log_model=True,
    log_prediction_images=True,
    prediction_samples=2,
    registered_model_name=None,
)

result = run_training(smoke_config, enable_mlflow=True)
result

## Equivalent CLI Command

The notebook call above is equivalent to running the project entry point:

```bash
uv run python train.py \
  --config params.yaml \
  --synthetic \
  --model tiny \
  --max-samples 4 \
  --max-batches 1 \
  --checkpoint-path models/mlflow-synthetic-smoke.pt \
  --log-model
```

This is the command to use during demos when you want an MLflow run quickly.

## Real VOC Tracked Run

When you are ready to use the real dataset, remove `--synthetic` and keep the run tiny:

```bash
uv run python train.py \
  --config params.yaml \
  --max-samples 8 \
  --max-batches 2 \
  --epochs 3 \
  --device mps \
  --checkpoint-path models/voc-mlflow-demo.pt \
  --log-model
```

Use `--device cpu` if MPS is unavailable. This downloads VOC if needed, logs params/metrics/artifacts, stores a model artifact, and logs segmentation preview images.

## Pretrained Visual Sanity Check

A randomly initialized U-Net is useful for explaining the workflow, but the masks will not look meaningful after a tiny run.

Use a torchvision segmentation model with VOC-style pretrained weights when you want to inspect a plausible output quickly:

```bash
uv run python train.py \
  --config params.yaml \
  --model lraspp_mobilenet_v3_large \
  --pretrained \
  --epochs 0 \
  --max-samples 4 \
  --device cpu \
  --checkpoint-path models/voc-lraspp-pretrained-preview.pt \
  --tracking-uri sqlite:///mlflow.db \
  --log-every-n-batches 5
```

`--epochs 0` logs the pretrained prediction preview without changing the model.

## Longer Observable Run

When you want to watch training progress in the terminal and MLflow together:

```bash
uv run python train.py \
  --config params.yaml \
  --model lraspp_mobilenet_v3_large \
  --pretrained \
  --max-samples 128 \
  --max-batches 40 \
  --val-max-samples 128 \
  --val-max-batches 40 \
  --epochs 8 \
  --device mps \
  --checkpoint-path models/voc-lraspp-pretrained-longer.pt \
  --tracking-uri sqlite:///mlflow.db \
  --log-every-n-batches 5
```

The terminal prints run settings, MLflow run ID, batch loss, train/validation metric summaries, preview artifacts, and checkpoint logging.

## Optional Registration

Registration should be explicit because it creates a named model lifecycle object.

```bash
uv run python train.py \
  --config params.yaml \
  --synthetic \
  --model tiny \
  --max-samples 4 \
  --max-batches 1 \
  --checkpoint-path models/registered-smoke.pt \
  --log-model \
  --registered-model-name ipcv-voc-segmentation
```

Use this after the tracking run is working and the local MLflow backend is initialized.

## Start The MLflow UI

```bash
uv run mlflow ui \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns \
  --host 127.0.0.1 \
  --port 5000
```

Open `http://127.0.0.1:5000` and inspect the `ipcv-voc-demo` experiment.

## What To Inspect

In the MLflow UI, look for:

- `Parameters`: dataset, model, batch size, learning rate, max samples
- `Metrics`: `train_loss`, `train_pixel_accuracy`, `train_mean_iou`
- `Metrics`: `val_loss`, `val_pixel_accuracy`, `val_mean_iou`
- `Images`: `segmentation_preview` across epochs
- `Artifacts`: `predictions/epoch_000.png` style preview panels
- `Artifacts`: checkpoint `.pt` file
- `Models`: logged PyTorch model, when `--log-model` is enabled
- `Registered Models`: named model, when `--registered-model-name` is provided

## Mental Model

```{mermaid}
%%| eval: true
%%| echo: false
flowchart LR
  A[train.py] --> B[MLflow run]
  B --> C[Params]
  B --> D[Metrics]
  B --> E[Checkpoint artifact]
  B --> F[PyTorch model artifact]
  F --> G[Optional registered model]
```

MLflow is the project memory: it records what was trained and what artifact came out of it.